# AML Kafka Consumer

Lê em streaming do tópico Kafka `transactions`, aplica o `PipelineModel` treinado offline e grava
previsões em HDFS (Parquet, particionado pela hora de chegada).

Também escreve métricas agregadas e mostra previsões no console para a defesa.

**Pré-requisitos:** `aml_trainer.ipynb` executado e modelo persistido.

In [ ]:
import os
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField,
    TimestampType, IntegerType, StringType, DoubleType,
)
from pyspark.ml import PipelineModel

KAFKA_BOOTSTRAP = os.environ.get("KAFKA_BOOTSTRAP", "10.84.129.52:9092")
KAFKA_TOPIC     = os.environ.get("KAFKA_TOPIC", "transactions")
SPARK_KAFKA_PKG = os.environ.get("SPARK_KAFKA_PKG", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0")

HDFS_BASE   = "hdfs://10.84.129.52:9000/aulas/fabricio_moreira/project"
MODEL_PATH  = f"{HDFS_BASE}/model/rf_aml_pipeline"
PRED_PATH   = f"{HDFS_BASE}/stream/predictions_kafka"
METRICS_PATH= f"{HDFS_BASE}/stream/metrics_kafka"
CHKPT_PRED  = f"{HDFS_BASE}/stream/chkpt_kafka_predictions"
CHKPT_METR  = f"{HDFS_BASE}/stream/chkpt_kafka_metrics"

spark = SparkSession.builder \
    .appName("AML Kafka Consumer") \
    .config("spark.jars.packages", SPARK_KAFKA_PKG) \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print(f"Kafka bootstrap: {KAFKA_BOOTSTRAP}  topic: {KAFKA_TOPIC}")

In [ ]:
model = PipelineModel.load(MODEL_PATH)
print("Modelo carregado.")

In [ ]:
msg_schema = StructType([
    StructField("Timestamp", TimestampType(), True),
    StructField("From Bank", IntegerType(), True),
    StructField("Account2", StringType(), True),
    StructField("To Bank", IntegerType(), True),
    StructField("Account4", StringType(), True),
    StructField("Amount Received", DoubleType(), True),
    StructField("Receiving Currency", StringType(), True),
    StructField("Amount Paid", DoubleType(), True),
    StructField("Payment Currency", StringType(), True),
    StructField("Payment Format", StringType(), True),
    StructField("Is Laundering", IntegerType(), True),
])

raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP) \
    .option("subscribe", KAFKA_TOPIC) \
    .option("startingOffsets", "latest") \
    .option("failOnDataLoss", "false") \
    .load()

parsed = raw.select(
    F.col("timestamp").alias("kafka_ts"),
    F.from_json(F.col("value").cast("string"), msg_schema).alias("d"),
).select("kafka_ts", "d.*")

In [ ]:
# Feature engineering — réplica exacta da do trainer
def add_features(d):
    return (
        d.withColumn("log_amt_paid", F.log1p(F.col("Amount Paid")))
         .withColumn("log_amt_recv", F.log1p(F.col("Amount Received")))
         .withColumn("amt_diff", F.abs(F.col("Amount Paid") - F.col("Amount Received")))
         .withColumn("same_bank", (F.col("From Bank") == F.col("To Bank")).cast("int"))
         .withColumn("ccy_mismatch", (F.col("Receiving Currency") != F.col("Payment Currency")).cast("int"))
         .withColumn("hour", F.hour("Timestamp"))
    )

featured = add_features(parsed).na.fill({"hour": 0})

scored = model.transform(featured) \
    .select(
        "kafka_ts", "Timestamp", "From Bank", "To Bank",
        "Amount Paid", "Receiving Currency", "Payment Format",
        F.col("Is Laundering").alias("truth"),
        F.col("prediction").cast("int").alias("prediction"),
        F.col("probability"),
    )

In [ ]:
# Sink 1: previsões em parquet (idempotente via checkpoint)
pred_query = scored \
    .withColumn("ingest_hour", F.date_format("kafka_ts", "yyyy-MM-dd-HH")) \
    .writeStream \
    .outputMode("append") \
    .format("parquet") \
    .partitionBy("ingest_hour") \
    .option("path", PRED_PATH) \
    .option("checkpointLocation", CHKPT_PRED) \
    .trigger(processingTime="5 seconds") \
    .start()

print("Sink de previsões iniciado:", PRED_PATH)

In [ ]:
# Sink 2: métricas agregadas em windows de 30s (throughput + taxa de positivos)
metrics = scored \
    .withWatermark("kafka_ts", "1 minute") \
    .groupBy(F.window("kafka_ts", "30 seconds")) \
    .agg(
        F.count("*").alias("n_msgs"),
        F.sum("prediction").alias("n_pred_pos"),
        F.sum("truth").alias("n_truth_pos"),
    )

metr_query = metrics.writeStream \
    .outputMode("append") \
    .format("parquet") \
    .option("path", METRICS_PATH) \
    .option("checkpointLocation", CHKPT_METR) \
    .trigger(processingTime="10 seconds") \
    .start()

print("Sink de métricas iniciado:", METRICS_PATH)

In [ ]:
# Sink 3: consola — útil para a defesa
console_query = scored.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .option("numRows", 20) \
    .trigger(processingTime="10 seconds") \
    .start()

spark.streams.awaitAnyTermination()

In [ ]:
# Parar manualmente (executar quando a demo terminar)
for q in spark.streams.active:
    print("A parar:", q.name or q.id)
    q.stop()